# Homework 1 — The Complete Data Science Workflow
## Predicting COVID-19 Mortality Risk

**Course:** MSIS 522  
**Dataset:** COVID-19 Patient Mortality (Mexican Government)  
**Task:** Binary Classification — Predict whether a COVID-19 patient will die (`DEATH = 1`) or survive (`DEATH = 0`)

---
# Part 1: Descriptive Analytics (25 pts)
---

## 1.1 Dataset Introduction (5 pts)

This dataset originates from Mexico's national COVID-19 epidemiological surveillance system. Each row represents one patient and the columns capture demographic information (sex, age, pregnancy status), hospitalization status, COVID-19 test result, and a comprehensive set of pre-existing conditions — diabetes, COPD, asthma, hypertension, obesity, cardiovascular disease, renal disease, immunosuppression, and tobacco use.

**Prediction Target:** `DEATH` — a binary variable indicating whether the patient died (1) or survived (0).

**Why this matters:** Predicting mortality risk from easily obtainable patient attributes (age, sex, comorbidities, hospitalization status) could help hospitals triage patients more effectively, allocate ICU beds and ventilators to those most at risk, and ultimately save lives — especially critical during pandemic surges when healthcare resources are stretched thin.

**Basic statistics:**
- Raw dataset: ~1,021,977 rows × 17 columns
- After balanced sampling: **10,000 rows** (5,000 died + 5,000 survived)
- Features: 15 binary (0/1) + 1 continuous (AGE, range 0–105) + 1 binary target (DEATH)
- No missing values in the cleaned dataset

In [ ]:
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')  # non-interactive backend for saving
import matplotlib.pyplot as plt
import seaborn as sns
import warnings, os, joblib, json

warnings.filterwarnings('ignore')
sns.set_style('whitegrid')
plt.rcParams['figure.dpi'] = 120

os.makedirs('models', exist_ok=True)
os.makedirs('plots', exist_ok=True)

In [ ]:
# ==== Load full dataset (only needed if you want to see the raw size) ====
# Uncomment for Colab:
# !wget -q -O covid.csv 'https://drive.google.com/uc?id=1R-GDTtX0l38JYlPaG7f8eKx3D6pN-CKE'
# data = pd.read_csv('covid.csv', usecols=lambda c: c != 'Unnamed: 0')
# print(f'Full dataset shape: {data.shape}')

# ==== Load the balanced 10K-row dataset ====
df = pd.read_csv('covid_balanced.csv')
print(f'Balanced dataset shape: {df.shape}')
print(f'\nColumn types:\n{df.dtypes}')
print(f'\nMissing values: {df.isnull().sum().sum()}')
df.head()

In [ ]:
df.describe()

## 1.2 Target Distribution (5 pts)

In [ ]:
fig, ax = plt.subplots(figsize=(6, 4))
counts = df['DEATH'].value_counts().sort_index()
bars = ax.bar(['Survived (0)', 'Died (1)'], counts.values,
              color=['#2ecc71', '#e74c3c'], edgecolor='black')
for bar, count in zip(bars, counts.values):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 50,
            str(count), ha='center', va='bottom', fontweight='bold', fontsize=12)
ax.set_title('Target Variable Distribution (DEATH)', fontsize=14, fontweight='bold')
ax.set_ylabel('Count')
plt.tight_layout()
plt.savefig('plots/target_distribution.png', bbox_inches='tight')
plt.show()

print(f'Survived: {counts[0]} | Died: {counts[1]}')
print('The dataset is perfectly balanced after deliberate sampling (50/50).')

**Interpretation:** After balanced sampling, we have exactly 5,000 patients in each class. The original dataset was heavily imbalanced (far more survivors than deaths), so this deliberate balancing ensures our models are not biased toward predicting survival. With a balanced dataset, accuracy is a reasonable metric, though we will still track F1, precision, recall, and AUC-ROC for a complete picture.

## 1.3 Feature Distributions and Relationships (10 pts)

In [ ]:
# ---- Visualization 1: Age Distribution by Outcome ----
fig, ax = plt.subplots(figsize=(8, 5))
for val, color, label in [(0, '#2ecc71', 'Survived'), (1, '#e74c3c', 'Died')]:
    subset = df[df['DEATH'] == val]['AGE']
    ax.hist(subset, bins=40, alpha=0.55, color=color, label=label, edgecolor='white')
ax.set_title('Age Distribution by Survival Outcome', fontsize=14, fontweight='bold')
ax.set_xlabel('Age'); ax.set_ylabel('Count'); ax.legend()
plt.tight_layout()
plt.savefig('plots/age_distribution_by_outcome.png', bbox_inches='tight')
plt.show()

**Interpretation:** Age is one of the strongest discriminating features. Patients who died are substantially older, with the mortality distribution peaking around ages 60–75. Survivors have a broader, younger distribution peaking around 35–45. This aligns with established medical evidence that advanced age is a primary COVID-19 risk factor.

In [ ]:
# ---- Visualization 2: Boxplot of Age by Outcome ----
fig, ax = plt.subplots(figsize=(6, 5))
sns.boxplot(data=df, x='DEATH', y='AGE',
            palette={0: '#2ecc71', 1: '#e74c3c'}, ax=ax)
ax.set_xticklabels(['Survived', 'Died'])
ax.set_title('Age Distribution: Survived vs Died', fontsize=14, fontweight='bold')
ax.set_xlabel('Outcome'); ax.set_ylabel('Age')
plt.tight_layout()
plt.savefig('plots/age_boxplot.png', bbox_inches='tight')
plt.show()
print(f"Median age — Survived: {df[df['DEATH']==0]['AGE'].median():.0f}")
print(f"Median age — Died:     {df[df['DEATH']==1]['AGE'].median():.0f}")

**Interpretation:** The boxplot confirms a large age gap between outcomes. The median age of deceased patients is roughly 15–20 years higher than that of survivors. The interquartile range for the deceased group is concentrated in the 55–75 band, while survivors show a wider spread skewing younger. Age alone has strong predictive power for mortality.

In [ ]:
# ---- Visualization 3: Mortality Rate by Comorbidity ----
comorbidities = ['PNEUMONIA','DIABETES','COPD','ASTHMA','IMMUNOSUPPRESSION',
                 'HYPERTENSION','CARDIOVASCULAR','RENAL_CHRONIC','OBESITY','TOBACCO']
mort = []
for c in comorbidities:
    mort.append({'Comorbidity': c.replace('_',' ').title(),
                 'With': df[df[c]==1]['DEATH'].mean()*100,
                 'Without': df[df[c]==0]['DEATH'].mean()*100})
mr = pd.DataFrame(mort)

fig, ax = plt.subplots(figsize=(10, 6))
x = np.arange(len(mr))
w = 0.35
ax.bar(x - w/2, mr['With'],    w, label='With Condition',    color='#e74c3c', edgecolor='black')
ax.bar(x + w/2, mr['Without'], w, label='Without Condition', color='#2ecc71', edgecolor='black')
ax.set_ylabel('Mortality Rate (%)')
ax.set_title('Mortality Rate by Pre-existing Condition', fontsize=14, fontweight='bold')
ax.set_xticks(x); ax.set_xticklabels(mr['Comorbidity'], rotation=45, ha='right')
ax.legend(); ax.axhline(50, color='gray', ls='--', alpha=.4)
plt.tight_layout()
plt.savefig('plots/mortality_by_comorbidity.png', bbox_inches='tight')
plt.show()

**Interpretation:** Pneumonia stands out with the largest mortality gap — patients with pneumonia die at a much higher rate than those without. COPD, renal chronic disease, and cardiovascular conditions also show elevated mortality. Interestingly, asthma and obesity show comparatively smaller differences, suggesting they are weaker standalone predictors of COVID-19 death. This chart is directly actionable for clinicians prioritizing patient care.

In [ ]:
# ---- Visualization 4: Hospitalization × Pneumonia Interaction ----
fig, ax = plt.subplots(figsize=(7, 5))
ct = df.groupby(['HOSPITALIZED','PNEUMONIA'])['DEATH'].mean().reset_index()
ct['Group'] = ct.apply(lambda r: f"Hosp={int(r.HOSPITALIZED)}, Pneu={int(r.PNEUMONIA)}", axis=1)
colors = ['#2ecc71','#f39c12','#e67e22','#e74c3c']
bars = ax.bar(ct['Group'], ct['DEATH']*100, color=colors, edgecolor='black')
for bar, val in zip(bars, ct['DEATH']*100):
    ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+1,
            f'{val:.1f}%', ha='center', fontweight='bold')
ax.set_ylabel('Mortality Rate (%)')
ax.set_title('Mortality Rate: Hospitalization × Pneumonia', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('plots/hosp_pneumonia_interaction.png', bbox_inches='tight')
plt.show()

**Interpretation:** This reveals a critical interaction effect. Patients who are both hospitalized AND have pneumonia show the highest mortality, while non-hospitalized patients without pneumonia show the lowest. The combination is a far stronger predictor than either factor alone — these features interact multiplicatively in driving risk.

In [ ]:
# ---- Visualization 5: Violin — Age by Sex and Outcome ----
fig, ax = plt.subplots(figsize=(8, 5))
tmp = df.copy()
tmp['Sex'] = tmp['SEX'].map({0:'Male',1:'Female'})
tmp['Outcome'] = tmp['DEATH'].map({0:'Survived',1:'Died'})
sns.violinplot(data=tmp, x='Sex', y='AGE', hue='Outcome', split=True,
               palette={'Survived':'#2ecc71','Died':'#e74c3c'}, ax=ax)
ax.set_title('Age Distribution by Sex and Outcome', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('plots/violin_age_sex_outcome.png', bbox_inches='tight')
plt.show()

**Interpretation:** For both sexes, mortality is concentrated among older patients. The 'Died' distribution is clearly skewed toward higher ages regardless of sex. Males who died appear to have a slightly wider age range, potentially indicating that males face elevated risk at somewhat younger ages — consistent with literature showing sex-based differences in COVID-19 vulnerability.

## 1.4 Correlation Heatmap (5 pts)

In [ ]:
fig, ax = plt.subplots(figsize=(12, 10))
corr = df.corr()
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='RdBu_r',
            center=0, vmin=-1, vmax=1, square=True, linewidths=.5, ax=ax)
ax.set_title('Correlation Heatmap — All Features', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('plots/correlation_heatmap.png', bbox_inches='tight')
plt.show()

print('\nStrongest correlations with DEATH:')
print(corr['DEATH'].drop('DEATH').abs().sort_values(ascending=False).head(8).to_string())

**Interpretation:** HOSPITALIZED and PNEUMONIA have the strongest positive correlations with DEATH — clinically intuitive since sicker patients require hospitalization and develop pneumonia. AGE also shows a meaningful positive correlation. Among comorbidities, DIABETES, HYPERTENSION, and RENAL_CHRONIC are moderately correlated with mortality. There is notable inter-feature correlation between DIABETES and HYPERTENSION (common comorbidity pairing) and between HOSPITALIZED and PNEUMONIA. Tree-based models handle this multicollinearity gracefully; logistic regression may be somewhat affected.

---
# Part 2: Predictive Analytics (45 pts)
---

## 2.1 Data Preparation

**Preprocessing decisions:**
- **Features (X):** All 16 columns except DEATH
- **Target (y):** DEATH (binary)
- **Split:** 70/30 train/test, `random_state=42`
- **Scaling:** StandardScaler applied for Logistic Regression and Neural Network (tree models don't need it)
- **Encoding:** Not needed — all features are already numeric
- **Missing values:** None present

In [ ]:
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, roc_auc_score, roc_curve,
                             confusion_matrix, classification_report)

X = df.drop(columns='DEATH')
y = df['DEATH']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42
)
print(f'Train: {X_train.shape[0]} samples | Test: {X_test.shape[0]} samples')

# Scaled versions for Logistic Regression & Neural Network
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled  = scaler.transform(X_test)

# Save scaler and feature names for Streamlit
joblib.dump(scaler, 'models/scaler.pkl')
joblib.dump(list(X.columns), 'models/feature_names.pkl')
print('Scaler and feature names saved.')

In [ ]:
# Helper: evaluate a model and return metrics dict
def evaluate_model(name, model, X_te, y_te, use_proba=True):
    preds = model.predict(X_te)
    # Handle Keras models returning probabilities from predict
    if hasattr(preds, 'shape') and len(preds.shape) > 1:
        preds_flat = (preds.ravel() > 0.5).astype(int)
    elif preds.dtype == float or (hasattr(preds, 'dtype') and np.issubdtype(preds.dtype, np.floating)):
        preds_flat = (preds.ravel() > 0.5).astype(int)
    else:
        preds_flat = preds
    
    if use_proba:
        if hasattr(model, 'predict_proba'):
            proba = model.predict_proba(X_te)[:, 1]
        else:
            proba = model.predict(X_te).ravel()
    else:
        proba = preds_flat.astype(float)
    
    metrics = {
        'Model': name,
        'Accuracy':  round(accuracy_score(y_te, preds_flat), 4),
        'Precision': round(precision_score(y_te, preds_flat), 4),
        'Recall':    round(recall_score(y_te, preds_flat), 4),
        'F1':        round(f1_score(y_te, preds_flat), 4),
        'AUC-ROC':   round(roc_auc_score(y_te, proba), 4),
    }
    print(f"\n{'='*50}")
    print(f"{name} — Test Set Metrics")
    print(f"{'='*50}")
    for k, v in metrics.items():
        if k != 'Model':
            print(f'  {k:>10s}: {v}')
    print(f"\nConfusion Matrix:\n{confusion_matrix(y_te, preds_flat)}")
    return metrics, proba

all_metrics = []  # collects results for comparison

## 2.2 Logistic Regression Baseline (5 pts)

In [ ]:
from sklearn.linear_model import LogisticRegression

lr = LogisticRegression(max_iter=1000, random_state=42)
lr.fit(X_train_scaled, y_train)

lr_metrics, lr_proba = evaluate_model('Logistic Regression', lr, X_test_scaled, y_test)
all_metrics.append(lr_metrics)

joblib.dump(lr, 'models/logistic_regression.pkl')
print('\nModel saved.')

## 2.3 Decision Tree / CART (5 pts)

In [ ]:
from sklearn.tree import DecisionTreeClassifier, plot_tree

dt_params = {
    'max_depth': [4, 5, 6, 9],
    'min_samples_leaf': [40, 50, 100, 200]
}

dt_grid = GridSearchCV(
    DecisionTreeClassifier(random_state=42),
    dt_params, cv=5, scoring='f1', n_jobs=-1, return_train_score=True
)
dt_grid.fit(X_train, y_train)

print(f'Best params: {dt_grid.best_params_}')
print(f'Best CV F1:  {dt_grid.best_score_:.4f}')

dt_best = dt_grid.best_estimator_
dt_metrics, dt_proba = evaluate_model('Decision Tree', dt_best, X_test, y_test)
all_metrics.append(dt_metrics)

joblib.dump(dt_best, 'models/decision_tree.pkl')
joblib.dump(dt_grid.best_params_, 'models/dt_best_params.pkl')

In [ ]:
# Visualize the best decision tree
fig, ax = plt.subplots(figsize=(20, 10))
plot_tree(dt_best, feature_names=X.columns, class_names=['Survived','Died'],
          filled=True, rounded=True, fontsize=8, ax=ax)
ax.set_title(f'Best Decision Tree (depth={dt_best.get_depth()})', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('plots/decision_tree_viz.png', bbox_inches='tight', dpi=150)
plt.show()

In [ ]:
# GridSearch heatmap for Decision Tree
results_dt = pd.DataFrame(dt_grid.cv_results_)
pivot_dt = results_dt.pivot_table(
    values='mean_test_score',
    index='param_min_samples_leaf',
    columns='param_max_depth'
)
fig, ax = plt.subplots(figsize=(7, 5))
sns.heatmap(pivot_dt, annot=True, fmt='.4f', cmap='YlOrRd', ax=ax)
ax.set_title('Decision Tree CV F1 Scores', fontsize=13, fontweight='bold')
ax.set_ylabel('min_samples_leaf'); ax.set_xlabel('max_depth')
plt.tight_layout()
plt.savefig('plots/dt_gridsearch_heatmap.png', bbox_inches='tight')
plt.show()

## 2.4 Random Forest (10 pts)

In [ ]:
from sklearn.ensemble import RandomForestClassifier

rf_params = {
    'n_estimators': [25, 50, 100, 200],
    'max_depth': [3, 5, 8, 10]
}

rf_grid = GridSearchCV(
    RandomForestClassifier(random_state=42),
    rf_params, cv=5, scoring='f1', n_jobs=-1, return_train_score=True
)
rf_grid.fit(X_train, y_train)

print(f'Best params: {rf_grid.best_params_}')
print(f'Best CV F1:  {rf_grid.best_score_:.4f}')

rf_best = rf_grid.best_estimator_
rf_metrics, rf_proba = evaluate_model('Random Forest', rf_best, X_test, y_test)
all_metrics.append(rf_metrics)

joblib.dump(rf_best, 'models/random_forest.pkl')
joblib.dump(rf_grid.best_params_, 'models/rf_best_params.pkl')

In [ ]:
# GridSearch heatmap for Random Forest
results_rf = pd.DataFrame(rf_grid.cv_results_)
pivot_rf = results_rf.pivot_table(
    values='mean_test_score',
    index='param_n_estimators',
    columns='param_max_depth'
)
fig, ax = plt.subplots(figsize=(7, 5))
sns.heatmap(pivot_rf, annot=True, fmt='.4f', cmap='YlOrRd', ax=ax)
ax.set_title('Random Forest CV F1 Scores', fontsize=13, fontweight='bold')
ax.set_ylabel('n_estimators'); ax.set_xlabel('max_depth')
plt.tight_layout()
plt.savefig('plots/rf_gridsearch_heatmap.png', bbox_inches='tight')
plt.show()

In [ ]:
# ROC Curve — Random Forest
fpr_rf, tpr_rf, _ = roc_curve(y_test, rf_proba)
fig, ax = plt.subplots(figsize=(6, 5))
ax.plot(fpr_rf, tpr_rf, color='#e74c3c', lw=2,
        label=f'Random Forest (AUC = {rf_metrics["AUC-ROC"]:.4f})')
ax.plot([0,1],[0,1], 'k--', lw=1, alpha=.5)
ax.set_xlabel('False Positive Rate'); ax.set_ylabel('True Positive Rate')
ax.set_title('ROC Curve — Random Forest', fontsize=13, fontweight='bold')
ax.legend(loc='lower right')
plt.tight_layout()
plt.savefig('plots/roc_random_forest.png', bbox_inches='tight')
plt.show()

## 2.5 Boosted Trees — LightGBM (10 pts)

In [ ]:
import lightgbm as lgb

lgb_params = {
    'n_estimators': [50, 100, 200],
    'max_depth': [3, 4, 5, 6],
    'learning_rate': [0.01, 0.05, 0.1]
}

lgb_grid = GridSearchCV(
    lgb.LGBMClassifier(random_state=42, verbosity=-1),
    lgb_params, cv=5, scoring='f1', n_jobs=-1, return_train_score=True
)
lgb_grid.fit(X_train, y_train)

print(f'Best params: {lgb_grid.best_params_}')
print(f'Best CV F1:  {lgb_grid.best_score_:.4f}')

lgb_best = lgb_grid.best_estimator_
lgb_metrics, lgb_proba = evaluate_model('LightGBM', lgb_best, X_test, y_test)
all_metrics.append(lgb_metrics)

joblib.dump(lgb_best, 'models/lightgbm.pkl')
joblib.dump(lgb_grid.best_params_, 'models/lgb_best_params.pkl')

In [ ]:
# ROC Curve — LightGBM
fpr_lgb, tpr_lgb, _ = roc_curve(y_test, lgb_proba)
fig, ax = plt.subplots(figsize=(6, 5))
ax.plot(fpr_lgb, tpr_lgb, color='#3498db', lw=2,
        label=f'LightGBM (AUC = {lgb_metrics["AUC-ROC"]:.4f})')
ax.plot([0,1],[0,1], 'k--', lw=1, alpha=.5)
ax.set_xlabel('False Positive Rate'); ax.set_ylabel('True Positive Rate')
ax.set_title('ROC Curve — LightGBM', fontsize=13, fontweight='bold')
ax.legend(loc='lower right')
plt.tight_layout()
plt.savefig('plots/roc_lightgbm.png', bbox_inches='tight')
plt.show()

In [ ]:
# ---- Optimal Threshold Selection ----
thresholds = np.arange(0.0, 1.01, 0.01)
f1s, precs, recs, specs = [], [], [], []

for t in thresholds:
    preds_t = (lgb_proba >= t).astype(int)
    f1s.append(f1_score(y_test, preds_t, zero_division=0))
    precs.append(precision_score(y_test, preds_t, zero_division=0))
    recs.append(recall_score(y_test, preds_t, zero_division=0))
    tn, fp, fn, tp = confusion_matrix(y_test, preds_t, labels=[0,1]).ravel()
    specs.append(tn / (tn + fp) if (tn + fp) > 0 else 0)

best_thresh_idx = np.argmax(f1s)
best_thresh = thresholds[best_thresh_idx]

fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(thresholds, f1s,   label='F1 Score',    lw=2)
ax.plot(thresholds, precs, label='Precision',    lw=2)
ax.plot(thresholds, recs,  label='Recall',       lw=2)
ax.plot(thresholds, specs, label='Specificity',  lw=2)
ax.axvline(best_thresh, color='red', ls='--', label=f'Best F1 threshold = {best_thresh:.2f}')
ax.set_xlabel('Threshold'); ax.set_ylabel('Score')
ax.set_title('LightGBM — Metrics vs Classification Threshold', fontsize=13, fontweight='bold')
ax.legend(loc='center left'); ax.set_xlim(0, 1)
plt.tight_layout()
plt.savefig('plots/threshold_analysis.png', bbox_inches='tight')
plt.show()

print(f'Optimal threshold (max F1): {best_thresh:.2f}')
print(f'F1 at optimal threshold:    {f1s[best_thresh_idx]:.4f}')

## 2.6 Neural Network — MLP with Keras (10 pts)

In [ ]:
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

tf.random.set_seed(42)
np.random.seed(42)

model_nn = keras.Sequential([
    layers.Input(shape=(X_train_scaled.shape[1],)),
    layers.Dense(128, activation='relu'),
    layers.Dense(128, activation='relu'),
    layers.Dense(1, activation='sigmoid')
])

model_nn.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)

model_nn.summary()

In [ ]:
history = model_nn.fit(
    X_train_scaled, y_train,
    epochs=50,
    batch_size=64,
    validation_split=0.2,
    verbose=1
)

In [ ]:
# Plot training history
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(history.history['loss'], label='Train Loss')
axes[0].plot(history.history['val_loss'], label='Val Loss')
axes[0].set_title('Loss Curve', fontweight='bold')
axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('Loss'); axes[0].legend()

axes[1].plot(history.history['accuracy'], label='Train Accuracy')
axes[1].plot(history.history['val_accuracy'], label='Val Accuracy')
axes[1].set_title('Accuracy Curve', fontweight='bold')
axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('Accuracy'); axes[1].legend()

plt.suptitle('Neural Network Training History', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('plots/nn_training_history.png', bbox_inches='tight')
plt.show()

In [ ]:
# Evaluate Neural Network
nn_proba = model_nn.predict(X_test_scaled).ravel()
nn_preds = (nn_proba > 0.5).astype(int)

nn_metrics = {
    'Model': 'Neural Network',
    'Accuracy':  round(accuracy_score(y_test, nn_preds), 4),
    'Precision': round(precision_score(y_test, nn_preds), 4),
    'Recall':    round(recall_score(y_test, nn_preds), 4),
    'F1':        round(f1_score(y_test, nn_preds), 4),
    'AUC-ROC':   round(roc_auc_score(y_test, nn_proba), 4),
}
all_metrics.append(nn_metrics)

print('Neural Network — Test Set Metrics')
for k, v in nn_metrics.items():
    if k != 'Model': print(f'  {k:>10s}: {v}')
print(f"\nConfusion Matrix:\n{confusion_matrix(y_test, nn_preds)}")

# Save the model
model_nn.save('models/neural_network.keras')

# Also save training history for Streamlit
hist_dict = {k: [float(v) for v in vals] for k, vals in history.history.items()}
with open('models/nn_history.json', 'w') as f:
    json.dump(hist_dict, f)
print('Neural network and history saved.')

In [ ]:
# ROC Curve — Neural Network
fpr_nn, tpr_nn, _ = roc_curve(y_test, nn_proba)
fig, ax = plt.subplots(figsize=(6, 5))
ax.plot(fpr_nn, tpr_nn, color='#9b59b6', lw=2,
        label=f'Neural Network (AUC = {nn_metrics["AUC-ROC"]:.4f})')
ax.plot([0,1],[0,1], 'k--', lw=1, alpha=.5)
ax.set_xlabel('False Positive Rate'); ax.set_ylabel('True Positive Rate')
ax.set_title('ROC Curve — Neural Network', fontsize=13, fontweight='bold')
ax.legend(loc='lower right')
plt.tight_layout()
plt.savefig('plots/roc_neural_network.png', bbox_inches='tight')
plt.show()

## 2.7 Model Comparison Summary (5 pts)

In [ ]:
comparison_df = pd.DataFrame(all_metrics)
comparison_df = comparison_df.set_index('Model')
print(comparison_df.to_string())

# Save for Streamlit
comparison_df.to_csv('models/model_comparison.csv')

In [ ]:
# Bar chart — F1 comparison
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

colors = ['#1abc9c', '#3498db', '#e74c3c', '#9b59b6', '#f39c12']
models = comparison_df.index.tolist()

# F1 comparison
bars1 = axes[0].bar(models, comparison_df['F1'], color=colors[:len(models)], edgecolor='black')
for bar, val in zip(bars1, comparison_df['F1']):
    axes[0].text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.005,
                 f'{val:.4f}', ha='center', fontsize=9, fontweight='bold')
axes[0].set_title('F1 Score Comparison', fontsize=13, fontweight='bold')
axes[0].set_ylabel('F1 Score')
axes[0].tick_params(axis='x', rotation=20)
axes[0].set_ylim(0, 1.05)

# AUC-ROC comparison
bars2 = axes[1].bar(models, comparison_df['AUC-ROC'], color=colors[:len(models)], edgecolor='black')
for bar, val in zip(bars2, comparison_df['AUC-ROC']):
    axes[1].text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.005,
                 f'{val:.4f}', ha='center', fontsize=9, fontweight='bold')
axes[1].set_title('AUC-ROC Comparison', fontsize=13, fontweight='bold')
axes[1].set_ylabel('AUC-ROC')
axes[1].tick_params(axis='x', rotation=20)
axes[1].set_ylim(0, 1.05)

plt.suptitle('Model Performance Comparison', fontsize=15, fontweight='bold')
plt.tight_layout()
plt.savefig('plots/model_comparison_bars.png', bbox_inches='tight')
plt.show()

In [ ]:
# Combined ROC curves
fig, ax = plt.subplots(figsize=(7, 6))

fpr_lr, tpr_lr, _ = roc_curve(y_test, lr_proba)
fpr_dt, tpr_dt, _ = roc_curve(y_test, dt_proba)

ax.plot(fpr_lr,  tpr_lr,  lw=2, label=f'Logistic Regression (AUC={lr_metrics["AUC-ROC"]:.3f})')
ax.plot(fpr_dt,  tpr_dt,  lw=2, label=f'Decision Tree (AUC={dt_metrics["AUC-ROC"]:.3f})')
ax.plot(fpr_rf,  tpr_rf,  lw=2, label=f'Random Forest (AUC={rf_metrics["AUC-ROC"]:.3f})')
ax.plot(fpr_lgb, tpr_lgb, lw=2, label=f'LightGBM (AUC={lgb_metrics["AUC-ROC"]:.3f})')
ax.plot(fpr_nn,  tpr_nn,  lw=2, label=f'Neural Network (AUC={nn_metrics["AUC-ROC"]:.3f})')
ax.plot([0,1],[0,1], 'k--', lw=1, alpha=.4)

ax.set_xlabel('False Positive Rate'); ax.set_ylabel('True Positive Rate')
ax.set_title('ROC Curves — All Models', fontsize=14, fontweight='bold')
ax.legend(loc='lower right', fontsize=9)
plt.tight_layout()
plt.savefig('plots/roc_all_models.png', bbox_inches='tight')
plt.show()

### Model Comparison Discussion

The table above shows that the ensemble methods (Random Forest and LightGBM) generally outperform the simpler Logistic Regression baseline and the single Decision Tree, as expected. LightGBM tends to achieve the highest AUC-ROC and F1, benefiting from gradient boosting's sequential error-correction approach. The Neural Network performs competitively but does not dramatically outperform the tree-based ensembles on this structured tabular dataset — a well-known pattern in the ML literature.

**Trade-offs:** The Decision Tree offers the best interpretability (you can visualize and explain every split), but sacrifices some accuracy. Random Forest and LightGBM are strong performers but act as black boxes without additional explainability tools like SHAP. The Neural Network is the least interpretable and most computationally expensive, yet does not clearly dominate on this relatively small, tabular dataset. For a clinical deployment where explainability matters, LightGBM paired with SHAP explanations offers the best balance of performance and transparency.

---
# Part 3: Explainability — SHAP (10 pts)
---

In [ ]:
import shap

# Use the best tree-based model (LightGBM)
explainer = shap.TreeExplainer(lgb_best)
shap_values = explainer.shap_values(X_test)

# For binary classification LightGBM, shap_values may be a list [class_0, class_1]
if isinstance(shap_values, list):
    shap_vals = shap_values[1]  # SHAP values for the positive class (DEATH=1)
else:
    shap_vals = shap_values

print(f'SHAP values shape: {shap_vals.shape}')

In [ ]:
# ---- SHAP Summary Plot (Beeswarm) ----
fig, ax = plt.subplots(figsize=(10, 7))
shap.summary_plot(shap_vals, X_test, show=False)
plt.title('SHAP Summary Plot — LightGBM', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('plots/shap_summary.png', bbox_inches='tight', dpi=150)
plt.show()

In [ ]:
# ---- SHAP Bar Plot ----
fig, ax = plt.subplots(figsize=(10, 6))
shap.summary_plot(shap_vals, X_test, plot_type='bar', show=False)
plt.title('Mean |SHAP Value| — Feature Importance', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('plots/shap_bar.png', bbox_inches='tight', dpi=150)
plt.show()

In [ ]:
# ---- SHAP Waterfall Plot for a High-Risk Patient ----
# Find a patient who died and was predicted correctly with high confidence
high_risk_mask = (y_test == 1) & (lgb_proba > 0.9)
if high_risk_mask.sum() > 0:
    idx = y_test[high_risk_mask].index[0]
    idx_pos = list(y_test.index).index(idx)
else:
    # fallback: just pick first deceased patient
    idx_pos = list(y_test.values).index(1)

print(f'Waterfall plot for test sample index {idx_pos}')
print(f'Actual outcome: {y_test.iloc[idx_pos]} | Predicted probability: {lgb_proba[idx_pos]:.3f}')
print(f'Patient features:\n{X_test.iloc[idx_pos]}')

# Create Explanation object for the waterfall plot
exp = shap.Explanation(
    values=shap_vals[idx_pos],
    base_values=explainer.expected_value[1] if isinstance(explainer.expected_value, list) else explainer.expected_value,
    data=X_test.iloc[idx_pos].values,
    feature_names=list(X_test.columns)
)

fig, ax = plt.subplots(figsize=(10, 7))
shap.plots.waterfall(exp, show=False)
plt.title('SHAP Waterfall — High-Risk Patient', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('plots/shap_waterfall.png', bbox_inches='tight', dpi=150)
plt.show()

### SHAP Interpretation

**Strongest predictors of mortality:**
- **HOSPITALIZED** is by far the most important feature. Being hospitalized dramatically increases the model's predicted mortality risk, which makes clinical sense — hospitalization is both a marker of disease severity and a proxy for unmeasured clinical signs.
- **AGE** is the second most influential feature. Older age consistently pushes predictions toward higher mortality, consistent with the well-documented age-mortality relationship in COVID-19.
- **PNEUMONIA** ranks third. Developing pneumonia as a complication strongly increases predicted death risk.

**Direction of impact:**
- High values of HOSPITALIZED, AGE, and PNEUMONIA push predictions toward DEATH=1 (red dots on the right side of the beeswarm).
- Comorbidities like DIABETES, HYPERTENSION, and RENAL_CHRONIC have moderate positive effects on mortality prediction.
- Features like ASTHMA, PREGNANCY, and TOBACCO have relatively small SHAP values, suggesting limited predictive power in this dataset.

**Clinical utility:** These insights directly support clinical decision-making. A triage system could flag hospitalized patients over 60 with pneumonia as highest priority for ICU resources. The SHAP waterfall plot allows clinicians to see exactly *why* a specific patient is flagged as high-risk, building trust in the model's recommendations.

In [ ]:
# Save SHAP values and expected value for Streamlit
np.save('models/shap_values.npy', shap_vals)
ev = explainer.expected_value
if isinstance(ev, list):
    ev = ev[1]
joblib.dump(float(ev), 'models/shap_expected_value.pkl')
print('SHAP artifacts saved.')

---
## Save all hyperparameters for Streamlit display
---

In [ ]:
best_params_all = {
    'Logistic Regression': {'C': 1.0, 'max_iter': 1000, 'solver': 'lbfgs'},
    'Decision Tree': dt_grid.best_params_,
    'Random Forest': rf_grid.best_params_,
    'LightGBM': lgb_grid.best_params_,
    'Neural Network': {'hidden_layers': '128-128', 'activation': 'relu',
                       'optimizer': 'adam', 'epochs': 50, 'batch_size': 64}
}
joblib.dump(best_params_all, 'models/best_params_all.pkl')

print('All hyperparameters saved.')
for name, params in best_params_all.items():
    print(f'  {name}: {params}')

In [ ]:
# Final check: list all saved files
print('\n=== Saved Models ===')
for f in sorted(os.listdir('models')):
    size = os.path.getsize(f'models/{f}') / 1024
    print(f'  {f:40s} ({size:.1f} KB)')

print('\n=== Saved Plots ===')
for f in sorted(os.listdir('plots')):
    size = os.path.getsize(f'plots/{f}') / 1024
    print(f'  {f:40s} ({size:.1f} KB)')

print('\nDone! All models and plots saved. Ready for Streamlit deployment.')